In [79]:
import pandas as pd
import numpy as np
import datetime 
import requests
from collections import defaultdict
from pyecharts.charts import Sankey
from pyecharts import options as opts

In [3]:
date1 = (datetime.date.today() - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

In [4]:
date2 = (datetime.date.today() - datetime.timedelta(days=14)).strftime('%Y-%m-%d')

In [16]:
date_list = [f'"{date2}"',f'"{date1}"']


In [21]:
date_list

'["2024-09-27","2024-10-10"]'

In [18]:
date_list = str(date_list).replace("'",'')
date_list = str(date_list).replace('"\'','"')
date_list = str(date_list).replace(' ','')

In [70]:
# 首先获取过去14日 大R、中R的付费数据
url = 'https://dash.engageminds.ai/api/oka/analysis/v1/event/report'
headers = {
        'Content-Type': 'application/json;charset=UTF-8'
}
body = {"appKey":"P3ewIzs9gigHVSRLXqvF4xjubPwDz5MZ",
        "pubAppId":175680,
        "fomulars":"[]",
        "groups":"[{\"id\":\"66530ff0-6a88-4235-89d1-5d19b72916dd\",\"display\":\"用户唯一id\",\"displayEn\":\"SSID\",\"value\":\"$ssid\",\"propType\":1,\"type\":1,\"category\":0,\"dataType\":2},{\"id\":\"98415343-0fc0-4000-91a5-28a1b8c8783d\",\"display\":\"事件发生时间\",\"displayEn\":\"事件发生时间\",\"value\":\"#vp@event_date\",\"propType\":0,\"type\":2,\"virtualSql\":\"event.$sts\",\"category\":0,\"dataType\":3},{\"id\":\"0448b565-71a7-41e6-b0f7-15b0137c7ae6\",\"display\":\"googleShopID\",\"displayEn\":\"googleShopID\",\"value\":\"googleShopID\",\"propType\":0,\"type\":0,\"category\":0,\"dataType\":2},{\"id\":\"c1b3ef6c-f6a8-4ec1-bff1-5ddf4ccc6cdf\",\"display\":\"转换为统一币种后收入\",\"displayEn\":\"App Currency Revenue\",\"value\":\"#em_revenue_ex\",\"propType\":1,\"type\":0,\"category\":0,\"dataType\":1}]",
        "tz":0,"dateType":2,"lang":"cn",
        "dateValue":"[\"Last 14 Days\",\"\"]",
        "events":"[{\"id\":\"5032b2b3-62a3-4adb-b7d8-85e217685d33\",\"f\":[],\"eid\":\"In_appPurchases_BuySuccess\",\"display\":\"内购成功\",\"displayEn\":\"内购成功\",\"type\":1,\"countType\":{\"value\":\"total\",\"name\":\"总次数\",\"nameEn\":\"Total\"}}]","filters":"[{\"id\":\"f37ef1e3-d68e-46db-801c-6186c4d8dee1\",\"eid\":\"\",\"f\":[{\"id\":\"0b849a54-11e1-42d9-93ae-8c5442aa7147\",\"f\":{},\"columnName\":\"$fit\",\"display\":\"首次安装App时间\",\"displayEn\":\"First Install App Time\",\"selectType\":\"Date\",\"symbolType\":3,\"propType\":1,\"type\":1,\"category\":0,\"calcuSymbol\":\"Between\",\"symbolId\":\"C01\",\"ftv\":%s,\"showCondition\":false,\"d\":1}]}]"% date_list}

In [71]:
data = requests.post(url, headers=headers, json=body)

In [81]:
data.json()['data']['data']

[{'$name': 'A:内购成功-总次数:3WIxaZ6Rfxxx2tv8yayjno,1728597094963,tales_award_2,5.7878',
  '#em_revenue_ex': '5.7878',
  '2024-09-27': 0,
  '2024-09-28': 0,
  '2024-09-29': 0,
  'sum': 1,
  'ssid': '3WIxaZ6Rfxxx2tv8yayjno',
  'googleShopID': 'tales_award_2',
  'total': 1,
  'avg': 0.07,
  '2024-10-09': 0,
  '2024-10-07': 0,
  '2024-10-08': 0,
  '2024-10-05': 0,
  '2024-10-06': 0,
  '2024-10-03': 0,
  '2024-10-04': 0,
  '2024-10-01': 0,
  '2024-10-02': 0,
  '2024-09-30': 0,
  'i': 1,
  'gv': '3WIxaZ6Rfxxx2tv8yayjno,1728597094963,tales_award_2,5.7878',
  'metric': 'A:内购成功-总次数',
  '#vp@event_date': '1728597094963',
  '2024-10-10': 1},
 {'$name': 'A:内购成功-总次数:2QdElLwWjgpkXfNE1JkpBZ,1727957762390,tales_highvalue_gift01,2.0538',
  '#em_revenue_ex': '2.0538',
  '2024-09-27': 0,
  '2024-09-28': 0,
  '2024-09-29': 0,
  'sum': 1,
  'ssid': '2QdElLwWjgpkXfNE1JkpBZ',
  'googleShopID': 'tales_highvalue_gift01',
  'total': 1,
  'avg': 0.07,
  '2024-10-09': 0,
  '2024-10-07': 0,
  '2024-10-08': 0,
  '2024-1

In [89]:
# 获取字典中的值
df_res=pd.DataFrame()
for i in data.json()['data']['data']:
    date_entries = {k: v for k, v in i.items() if 'gv' in k}
    df = pd.DataFrame(date_entries.items(), columns=['gv','nf_info'])
    df['金额'] = eval(i['#em_revenue_ex'])
    if df_res.shape[0] == 0:
        df_res = df
    else:
        df_res = pd.concat((df_res,df))
    
    

In [90]:
df_res['nginfo'] = df_res['nf_info'].str.split(',')

In [91]:
df_res['ssid'] = df_res.apply(lambda x:x['nginfo'][0],axis=1)
df_res['ng_time'] = df_res.apply(lambda x:x['nginfo'][1],axis=1)
df_res['shopid'] = df_res.apply(lambda x:x['nginfo'][2],axis=1)

In [92]:
df_res['总金额'] = df_res.groupby(by=['ssid'])['金额'].transform('sum')

In [107]:
df_resRR = df_res[['ssid','ng_time','shopid']]

In [106]:
df_res.query('总金额>100')['ssid'].nunique()

17

In [142]:
df_res['购买时间'] =pd.to_datetime(df_res['ng_time'],unit = 'ms')

C:\Users\admin\AppData\Local\Temp\ipykernel_12100\4104341832.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df_res['购买时间'] =pd.to_datetime(df_res['ng_time'],unit = 'ms')


In [143]:
# 获取购买顺序
df_res = df_res.sort_values(by=['ssid', '购买时间'])

In [151]:
df_res=df_res.merge(shop_info,how='left',on='shopid')

In [110]:
df_res[['ssid','shopid','购买时间']].to_csv('../临时文件/大R购买记录.csv',index=False)

In [138]:
df_resRR['ssid'].nunique()

17

In [152]:
# 创建一个 DataFrame 来存储前五次购买的路径数据
purchase_data = []

# 为每个用户生成前五次购买路径（如果存在）
for user_id, group in df_res.groupby('ssid'):
    # 获取用户的购买记录
    purchase_list = group['分类'].tolist()
    # 只处理前5次购买，如果不足5次，则只处理存在的购买记录
    num_purchases = min(len(purchase_list), 5)
    for i in range(num_purchases - 1):
        first_purchase = purchase_list[i]
        if len(purchase_list)-1 >=i+1 :
            second_purchase = purchase_list[i+1]
        else :
            second_purchase = '-'
        if len(purchase_list)-1 >=i+2:
            third_purchase = purchase_list[i+2]
        else:
            third_purchase = '-'
        if len(purchase_list)-1 >=i+3:
            fourth_purchase = purchase_list[i+3]
        else:
            fourth_purchase= '-'
        if len(purchase_list)-1 >= i+4:
            # print(len(purchase_list),i+4)
            fifth_purchase = purchase_list[i+4]
        else:
            fifth_purchase = '-'
        
        
    purchase_data.append({
            '用户ID': user_id,
            '第一次购买': first_purchase,
            '第二次购买': second_purchase,
            '第三次购买': third_purchase,
            '第四次购买': fourth_purchase,
            '第五次购买': fifth_purchase,
    })

# 转换为 DataFrame 并统计每个购买路径的出现次数
purchase_df = pd.DataFrame(purchase_data)

# 对照表
purchase_count = purchase_df.groupby(['第一次购买', '第二次购买','第三次购买', '第四次购买', '第五次购买']).size().reset_index(name='人数')

In [153]:
purchase_count.to_excel('../临时文件/所有用户付费路径.xlsx',index=False)

In [149]:
shop_info = pd.read_excel('../临时文件/商品对照表.xlsx')[['谷歌商店ID','分类']]

In [150]:
shop_info.columns = ['shopid','分类']